# Imports

In [ ]:
import numpy as np
from torch.utils.data import Dataset


# Data

In [ ]:
from __future__ import annotations
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from typing import Dict, List, Tuple, Optional

class KInterpPerCompoundDataset(Dataset):
    """
    One dataset item = one compound (SMILES). For each compound, __getitem__
    generates exactly K synthetic (interpolated) samples by:
      1) picking a concentration for that compound,
      2) picking an adjacent time interval (t[i], t[i+1]),
      3) sampling t ~ Uniform(t[i], t[i+1]) and linearly interpolating OD,
      4) deriving the 'active' label (recommended: from OD threshold).

    df columns expected: ['Compound', 'Smiles', 'Concentration', 'Timepoint', 'OD', 'is_Active']
    fingerprints: mapping Compound -> np.ndarray (float32) OR a DF with 'Compound' + fp columns
    Inputs to model are [fingerprint, Timepoint, Concentration] by default (customize as needed).

    Returns per item:
      x:   torch.float32 [K, F]  (features)
      y_r: torch.float32 [K]     (OD regression targets)
      y_c: torch.int64   [K]     (binary class indices)
      meta: dict with helpful context (compound name, per-sample conc/time, mask of augmented)
    """

    def __init__(
        self,
        df: pd.DataFrame,
        fingerprints: Dict[str, np.ndarray] | pd.DataFrame,
        *,
        k: int = 4,
        activity_from_od: bool = True,
        od_active_thresh: Optional[float] = None,  # required if activity_from_od=True
        include_real_fallback: bool = True,        # fill any “no-interval” cases with real points
        choose_conc: str = "uniform",              # 'uniform' or 'proportional' (by # intervals)
        feature_dtype: torch.dtype = torch.float32,
    ):
        assert k >= 1
        self.k = k
        self.activity_from_od = activity_from_od
        self.od_active_thresh = od_active_thresh
        self.include_real_fallback = include_real_fallback
        self.choose_conc = choose_conc
        self.feature_dtype = feature_dtype

        # --- Build per-compound trajectories grouped by concentration ---
        # Keep only necessary columns and sort by time within each (compound, conc)
        cols_needed = ["Compound", "Concentration", "Timepoint", "OD", "is_Active"]
        gdf = df[cols_needed].copy()
        gdf.sort_values(["Compound", "Concentration", "Timepoint"], inplace=True)

        # Per-compound index (dataset items)
        self.compounds: List[str] = list(gdf["Compound"].unique())

        # Map compound -> { conc: (t, od, y) }
        self.trajs: Dict[str, Dict[float, Tuple[np.ndarray, np.ndarray, np.ndarray]]] = {}
        for cmpd, gcmp in gdf.groupby("Compound", sort=False):
            per_conc: Dict[float, Tuple[np.ndarray, np.ndarray, np.ndarray]] = {}
            for conc, gc in gcmp.groupby("Concentration", sort=False):
                t = gc["Timepoint"].to_numpy(dtype=float)
                od = gc["OD"].to_numpy(dtype=float)
                y = gc["is_Active"].to_numpy(dtype=int)
                per_conc[float(conc)] = (t, od, y)
            self.trajs[cmpd] = per_conc

        # --- Fingerprints map ---
        if isinstance(fingerprints, pd.DataFrame):
            # Expect a 'Compound' column + fp columns
            fp_cols = [c for c in fingerprints.columns if c != "Compound"]
            self.fp_dim = len(fp_cols)
            self.fp_map = {
                row["Compound"]: row[fp_cols].to_numpy(dtype=np.float32)
                for _, row in fingerprints.iterrows()
            }
        else:
            # Dict[str, np.ndarray]
            self.fp_map = {k: np.asarray(v, dtype=np.float32) for k, v in fingerprints.items()}
            dims = {v.shape for v in self.fp_map.values()}
            assert len(dims) == 1, "All fingerprints must have the same shape"
            self.fp_dim = list(dims)[0][0]

        # Pre-allocate a working buffer shape (K, F) where F = fp_dim + 2 (time, conc)
        self.feat_dim = self.fp_dim + 2

    def __len__(self) -> int:
        return len(self.compounds)

    # ---- sampling helpers ----
    @staticmethod
    def _interp_linear(t0, t1, y0, y1, t_new):
        return y0 + (y1 - y0) * ((t_new - t0) / (t1 - t0))

    def _sample_one(self, cmpd: str) -> Tuple[np.ndarray, float, int, float]:
        """
        Sample ONE synthetic data point for this compound.
        Returns: (fp vec), time_new, cls_label, conc
        Also returns OD (for regression target).
        """
        conc_map = self.trajs[cmpd]
        # Gather concentrations that have >= 2 timepoints (interpolable)
        concs_interpolable = [c for c, (t, _, _) in conc_map.items() if len(t) >= 2]

        if len(concs_interpolable) == 0:
            # No interpolation possible; fall back to a real observation if allowed
            if not self.include_real_fallback:
                raise RuntimeError(f"No interpolable intervals and fallback disabled for {cmpd}")
            # choose any conc; pick a real row
            conc_choices = list(conc_map.keys())
            c = conc_choices[np.random.randint(len(conc_choices))]
            t, od, y = conc_map[c]
            j = np.random.randint(len(t))
            t_val, od_val, y_val = float(t[j]), float(od[j]), int(y[j])
            return self.fp_map[cmpd], t_val, y_val, float(c), od_val

        # Choose concentration
        if self.choose_conc == "proportional":
            # weight by number of intervals in each conc (len(t)-1)
            weights = np.array([max(0, len(conc_map[c][0]) - 1) for c in concs_interpolable], dtype=float)
            weights = weights / weights.sum()
            c = np.random.choice(concs_interpolable, p=weights)
        else:
            c = np.random.choice(concs_interpolable)

        t, od, y = conc_map[c]
        # Pick one adjacent interval uniformly
        i = np.random.randint(len(t) - 1)
        t0, t1 = t[i], t[i + 1]
        u = np.random.rand()
        t_new = t0 + u * (t1 - t0)
        od_new = self._interp_linear(t0, t1, od[i], od[i + 1], t_new)

        if self.activity_from_od:
            assert self.od_active_thresh is not None, "Provide od_active_thresh when activity_from_od=True"
            y_new = int(od_new <= self.od_active_thresh)
        else:
            # nearest neighbor label
            y_new = int(y[i] if (t_new - t0) <= (t1 - t0) else y[i + 1])

        return self.fp_map[cmpd], float(t_new), y_new, float(c), float(od_new)

    def __getitem__(self, idx: int):
        cmpd = self.compounds[idx]
        fp = self.fp_map[cmpd]  # (fp_dim,)

        # Generate exactly K samples
        X = np.empty((self.k, self.feat_dim), dtype=np.float32)
        times = np.empty((self.k,), dtype=np.float32)
        concs = np.empty((self.k,), dtype=np.float32)
        y_reg = np.empty((self.k,), dtype=np.float32)
        y_cls = np.empty((self.k,), dtype=np.int64)
        aug_mask = np.ones((self.k,), dtype=np.bool_)

        for j in range(self.k):
            try:
                fp_j, t_j, y_j, c_j, od_j = self._sample_one(cmpd)
                # fp_j should equal fp; but we re-use for clarity
            except RuntimeError:
                # If fallback disabled and no intervals, create NaNs (or you could raise)
                fp_j, t_j, y_j, c_j, od_j = fp, np.nan, 0, np.nan, np.nan
                aug_mask[j] = False

            # Features = [fp, t, c]  (customize if you encode time/conc elsewhere)
            X[j, : self.fp_dim] = fp_j
            X[j, self.fp_dim + 0] = t_j
            X[j, self.fp_dim + 1] = c_j
            times[j] = t_j
            concs[j] = c_j
            y_reg[j] = od_j
            y_cls[j] = y_j

        out = {
            "x": torch.from_numpy(X).to(self.feature_dtype),  # [K, F]
            "y_reg": torch.from_numpy(y_reg).to(torch.float32),  # [K]
            "y_cls": torch.from_numpy(y_cls).to(torch.long),     # [K]
            "meta": {
                "compound": cmpd,
                "time": torch.from_numpy(times),
                "conc": torch.from_numpy(concs),
                "augmented_mask": torch.from_numpy(aug_mask),
            },
        }
        return out

# --------- Collate: stack per-compound K-bundles into [B, K, F] (and optionally flatten) ---------

def collate_kinterp(batch: List[Dict], flatten: bool = True):
    # batch is a list of dicts from __getitem__
    x = torch.stack([b["x"] for b in batch], dim=0)           # [B, K, F]
    y_reg = torch.stack([b["y_reg"] for b in batch], dim=0)   # [B, K]
    y_cls = torch.stack([b["y_cls"] for b in batch], dim=0)   # [B, K]

    meta = {
        "compound": [b["meta"]["compound"] for b in batch],
        "time": torch.stack([b["meta"]["time"] for b in batch], dim=0),  # [B, K]
        "conc": torch.stack([b["meta"]["conc"] for b in batch], dim=0),  # [B, K]
        "augmented_mask": torch.stack([b["meta"]["augmented_mask"] for b in batch], dim=0),  # [B, K]
    }

    if flatten:
        B, K, F = x.shape
        x = x.view(B * K, F)
        y_reg = y_reg.view(B * K)
        y_cls = y_cls.view(B * K)
        meta["time"] = meta["time"].view(B * K)
        meta["conc"] = meta["conc"].view(B * K)
        meta["augmented_mask"] = meta["augmented_mask"].view(B * K)
        # 'compound' stays as list length B; optionally repeat if you need 1:1 mapping:
        # meta["compound_rep"] = sum([[c]*K for c in meta["compound"]], [])
    return {"x": x, "y_reg": y_reg, "y_cls": y_cls, "meta": meta}

# --------- Usage ---------

def seed_worker(_):
    seed = torch.initial_seed() % 2**32
    np.random.seed(seed)

# df_train: your long table with columns mentioned above
# fp_map: dict {compound -> fingerprint np.array(dtype=float32)}  OR a DF with 'Compound' + fp columns

# Example:
# train_ds = KInterpPerCompoundDataset(df_train, fp_map, k=8,
#                                      activity_from_od=True, od_active_thresh=0.25,
#                                      choose_conc="proportional")
# train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,
#                           num_workers=4, pin_memory=True,
#                           worker_init_fn=seed_worker,
#                           collate_fn=lambda b: collate_kinterp(b, flatten=True))
